# 06 · Analysis
Inference: seeded samples, truncation grids, interpolations. Evaluation: FID, precision/recall, equivariance, spectra.

In [ ]:
import os, sys
# ---- platform auto-detect: the same notebook runs on Colab and Kaggle ----
PLATFORM = "kaggle" if os.path.exists("/kaggle/input") else "colab"
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if (PLATFORM == "colab" or PLATFORM == "kaggle") and not os.path.exists("src"):
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/Ravikishore710/styleforge3-T.git"], check=True)
    os.chdir("styleforge3-T")
sys.path.insert(0, os.path.abspath("."))
!pip install -q -r requirements.txt
import tensorflow as tf
print("platform:", PLATFORM, "| TF:", tf.__version__,
      "| GPU:", tf.config.list_physical_devices("GPU"))
# Kaggle: enable GPU (Settings -> Accelerator -> GPU P100) and add the FFHQ
# dataset to /kaggle/input, or run scripts/prepare_ffhq.py --source folder.

In [ ]:
!python scripts/generate.py --config configs/ffhq_64.yaml --num-images 16 --seed 42
!python scripts/generate.py --config configs/ffhq_64.yaml --truncation-grid
!python scripts/generate.py --config configs/ffhq_64.yaml --interpolate

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for a, p in zip(ax, ['outputs/samples/seeded_42.png',
                     'outputs/samples/truncation_grid.png',
                     'outputs/samples/interpolation.png']):
    a.imshow(mpimg.imread(p)); a.axis('off'); a.set_title(p.split('/')[-1], fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
!python scripts/evaluate.py --config configs/ffhq_64.yaml --fid-images 2000 --skip-eq

In [ ]:
import json
print(json.dumps(json.load(open("outputs/metrics/metrics.json")), indent=2))

## Equivariance (the StyleGAN3-specific metric)
Small image counts keep it fast; use `--fid-images 500` with equivariance enabled for the full check.

In [ ]:
!python scripts/evaluate.py --config configs/ffhq_64.yaml --fid-images 500